<a href="https://colab.research.google.com/github/batrisyiasuh/batrisyia5842342-AnalyticsEngineering/blob/main/eisha_test%20practice%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [3]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [4]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [5]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [8]:
# 1. Basic Inspection: missing values & duplicate rows
for name, df in tables.items():
    print(f"=== Table: {name} {df.shape} ===")
    print("\nMissing values:")
    print(df.isna().sum())
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    print("-" * 40)

# 2. Categories & Key Checks (e.g., bike types, user types, zones)
print("=== Categories & Key Checks ===")
if 'bikes' in tables:
    print("Bike Types found:", tables['bikes']['bike_type'].unique())
    print("Bike Purchase Years:", tables['bikes']['purchase_year'].unique())

if 'trips' in tables:
    print("User Types found:", tables['trips']['user_type'].unique())

if 'stations' in tables:
    print("Zones found:", tables['stations']['zone'].unique())
print("-" * 40)

# 3. Numeric Oddities (e.g., duration bounds, capacity bounds)
print("=== Numeric Oddities ===")
if 'trips' in tables:
    print("Trip duration summary:")
    print(tables['trips']['duration_min'].describe())

if 'stations' in tables:
    print("Station capacity summary:")
    print(tables['stations']['capacity'].describe())
print("-" * 40)

# 4. Dates
print("=== Dates ===")
if 'trips' in tables:
    print("Trip start_time range:")
    print("Min:", tables['trips']['start_time'].min())
    print("Max:", tables['trips']['start_time'].max())
print("-" * 40)

# 5. Referential Integrity (Foreign Key checks)
print("=== Referential Integrity ===")
if 'trips' in tables and 'bikes' in tables:
    orphan_bikes = set(tables['trips']['bike_id']) - set(tables['bikes']['bike_id'])
    print(f"Trip bike_ids not matching bikes table: {len(orphan_bikes)} (IDs: {orphan_bikes})")

if 'trips' in tables and 'stations' in tables:
    orphan_start_stations = set(tables['trips']['start_station_id']) - set(tables['stations']['station_id'])
    orphan_end_stations = set(tables['trips']['end_station_id']) - set(tables['stations']['station_id'])
    print(f"Trip start_station_ids not matching stations: {len(orphan_start_stations)} (IDs: {orphan_start_stations})")
    print(f"Trip end_station_ids not matching stations: {len(orphan_end_stations)} (IDs: {orphan_end_stations})")
print("-" * 40)

=== Table: trips (612, 7) ===

Missing values:
trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

Duplicate rows: 12
----------------------------------------
=== Table: bikes (60, 3) ===

Missing values:
bike_id          0
bike_type        0
purchase_year    0
dtype: int64

Duplicate rows: 0
----------------------------------------
=== Table: stations (12, 4) ===

Missing values:
station_id      0
station_name    0
zone            0
capacity        1
dtype: int64

Duplicate rows: 0
----------------------------------------
=== Categories & Key Checks ===
Bike Types found: ['Standard' 'Electric ' 'electric' 'STANDARD' 'standard' 'Electric'
 'E-bike']
Bike Purchase Years: [2021 2022 2024 2023]
User Types found: ['Casual' 'CASUAL' 'Member' 'MEMBER' 'Casual ' 'member' 'casual']
Zones found: ['Centre' 'North' 'West' 'South' 'East']
--------------------------------------

,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member


ISO: 514 | dd/mm/yyyy: 98 | still unparseable: 0
Range: 2025-03-01 05:52:00 to 2025-03-31 21:54:00


# Data Understanding and Cleaning Decisions

## 1. Understanding the Tables

### What one row represents

- **`trips`**: One bike hire (journey), containing 612 rows. Primary key: `trip_id`.
- **`bikes`**: One physical bike, containing 60 rows. Primary key: `bike_id`. Contains `bike_type`.
- **`stations`**: One docking station, containing 12 rows. Primary key: `station_id`. Contains `zone`.

### Columns used in the analysis

The analysis focuses on bike type, starting station zone, and trip duration.

- `trips.duration_min`
- `trips.bike_id`
- `trips.start_station_id`
- `bikes.bike_type`
- `stations.zone`

The column `trip_id` will also be used to check for duplicate records.

### Columns excluded from the analysis

- `end_station_id`
- `user_type`
- `start_time` (used only for data validation)
- `purchase_year`
- `station_name`
- `capacity`

These columns are not directly relevant to the research question, which focuses on the relationship between bike type, starting station zone, and trip duration.

---

## 2. Data Quality Assessment

The following issues were identified during the initial data exploration.

| Issue | Evidence | Decision and Justification |
|---|---|---|
| Inconsistent `bike_type` labels | Seven variants: `Standard`, `standard`, `STANDARD`, `Electric`, `Electric `, `electric`, and `E-bike`. | Remove leading/trailing whitespace, convert labels to lowercase, and standardise `e-bike` to `electric`. This ensures consistent grouping into two categories. The assumption that `E-bike` means `Electric` will be stated explicitly. |
| Missing `duration_min` | 30 of 612 records (approximately 4.9%). | Exclude missing durations from average calculations because the values cannot be determined reliably. Report the number excluded and examine missingness across groups. |
| Extreme trip durations | Maximum of 2,880 minutes (48 hours), compared with a median of 16 minutes and a 75th percentile of 22 minutes. | Investigate the extreme values and exclude trips exceeding 120 minutes. These values may reflect bikes not being returned or logging errors. Report the threshold and compare the mean with the median. |
| Duplicate records | 12 exact duplicate rows in `trips`. | Remove exact duplicates to avoid double-counting journeys. Verify duplicate `trip_id` values before proceeding. |
| Dates stored as text | `start_time` contains ISO-format and `dd/mm/yyyy` dates. | Parse each format explicitly and combine the results. Use the parsed dates to validate the data range. |
| Missing station capacity | 1 of 12 stations has a missing `capacity`. | Leave unchanged because capacity is not required for this analysis. |
| Inconsistent `user_type` labels | Values such as `Casual`, `CASUAL`, and `Casual `. | Leave unchanged because `user_type` is outside the scope of the research question. Standardisation would be necessary if this variable were used. |

---

## 3. Data Quality Checks That Passed

The following checks indicate that the lookup tables and relationships are suitable for analysis.

- **No duplicate lookup records:** The `bikes` and `stations` tables contain no fully duplicated rows.
- **Unique keys:** `bike_id` is unique across the 60 bikes, and `station_id` is unique across the 12 stations.
- **Valid relationships:** Every `bike_id` and `start_station_id` referenced in `trips` matches a record in its respective lookup table. No orphan records were identified.
- **Consistent station zones:** The `zone` column contains five clean categories: Centre, North, West, South, and East.

These checks indicate that joining the three tables should not introduce row multiplication or lose records due to unmatched keys.

---

## 4. Detailed Cleaning Decisions

### 4.1 Duplicate Records

The `trips` table initially contains 612 rows but only 600 distinct `trip_id` values.

- There are 12 fully identical duplicate rows.
- The check for non-identical duplicates returned zero results.
- Every repeated `trip_id` corresponds to an exact copy of an existing record.

**Decision:** Remove exact duplicate rows, reducing the dataset from 612 to 600 records.

**Justification:** Each trip should be counted only once. Retaining identical copies would inflate journey counts and potentially bias the analysis. Since the duplicate records are identical, there is no ambiguity about which copy to retain.

The `bikes` and `stations` tables have unique keys, so joining them to the cleaned `trips` table should not multiply the number of records.

### 4.2 Date Formatting

The `start_time` column is stored as text in two formats:

- Approximately 514 records use ISO date formatting.
- Approximately 98 records use the `dd/mm/yyyy` format.

After parsing both formats explicitly, all dates fall within March 2025.

**Decision:** Parse each date format separately and combine the results into a consistent datetime column.

**Justification:** Although dates are not required for the main analysis, validating them helps confirm that the dataset covers the expected period and that the date values have been interpreted correctly.

### 4.3 Duration Outliers

Only three trips exceed 120 minutes:

- Two trips lasting exactly 1,440 minutes (24 hours).
- One trip lasting exactly 2,880 minutes (48 hours).

The 99th percentile is 52 minutes, indicating a substantial gap between typical trips and these extreme observations.

The exact multiples of 24 hours suggest possible unreturned bikes or default logging values rather than genuine journeys.

**Decision:** Exclude trips with durations exceeding 120 minutes. This removes three records, approximately 0.5% of the original 612 rows.

**Justification:** The threshold is supported by the gap in the duration distribution and the implausible, round-day durations. These observations can substantially inflate the mean.

The overall mean duration is approximately 27.9 minutes, compared with a median of 16 minutes. After removing the extreme observations, the mean falls to approximately 18 minutes, much closer to the median.

The median will also be reported alongside the mean as a robustness check because it is less sensitive to extreme values.

### 4.4 Missing Trip Durations

The `duration_min` column contains 30 missing values, representing approximately 4.9% of the original records.

**Decision:** Exclude records with missing durations from calculations of average trip duration.

**Justification:** Imputing missing values would introduce assumptions into the variable being measured. Instead, the number of missing values will be reported, and missingness will be examined across bike types and station zones to determine whether it is concentrated in particular groups.

---

## 5. Summary of the Cleaning Approach

The analysis will follow these steps:

1. Remove exact duplicate trip records.
2. Standardise `bike_type` labels by trimming whitespace, converting to lowercase, and mapping `e-bike` to `electric`.
3. Parse `start_time` using the appropriate date formats and validate the date range.
4. Exclude trips with durations exceeding 120 minutes.
5. Exclude missing `duration_min` values from average calculations and report their distribution across groups.
6. Join `trips` with `bikes` using `bike_id`.
7. Join the resulting table with `stations` using `start_station_id` and `station_id`.
8. Compare trip durations by bike type and starting station zone.
9. Report the mean and median, alongside group sizes and excluded-record counts, to support transparent interpretation.

**Overall approach:** Only data issues that could affect the analysis will be corrected or excluded. Unrelated variables will remain unchanged, and all cleaning decisions will be documented to ensure the analysis is transparent and reproducible.

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [14]:
#Do the missing durations cluster anywhere?
#If most are in one bike type or zone, dropping them could bias the average.
#How many bikes are in each type?
#This confirms the “e-bike” mapping didn’t produce something odd.

chk = (tables['trips'].drop_duplicates()
       .merge(bikes_cleaned[['bike_id','bike_type']], on='bike_id')
       .merge(tables['stations'][['station_id','zone']], left_on='start_station_id', right_on='station_id'))
chk['missing'] = chk['duration_min'].isna()
display(chk.groupby('bike_type')['missing'].agg(['sum','mean']))
display(chk.groupby('zone')['missing'].agg(['sum','mean']))
print(bikes_cleaned['bike_type'].value_counts())

,sum,mean
bike_type,,
electric,13,0.074713
standard,17,0.039906


,sum,mean
zone,,
Centre,16,0.050955
East,5,0.138889
North,1,0.015873
South,2,0.023256
West,6,0.059406


bike_type
standard    47
electric    13
Name: count, dtype: int64


### Decisions to Document in the Data Cleaning Notes

#### 1. Duplicate Trips
- **Problem:** 12 exact duplicate trips.
- **Action taken:** Removed duplicate rows, retaining one copy of each trip.
- **Justification:** Each trip should be counted only once. Since the duplicates were identical, there was no ambiguity about which record to retain.
- **Evidence:** The row count decreased from 612 to 600, matching the 600 distinct `trip_id` values.

#### 2. Inconsistent `bike_type` Labels
- **Problem:** Seven different spellings of bike types.
- **Action taken:** Removed leading/trailing whitespace, converted labels to lowercase, and mapped `e-bike` to `electric`.
- **Justification:** Without standardisation, electric bikes would be split into multiple groups.
- **Assumption:** `e-bike` refers to an electric bike.
- **Evidence:** Exactly two categories remain: `standard` and `electric`.

#### 3. Mixed Date Formats
- **Problem:** Dates were stored in ISO and `dd/mm/yyyy` formats.
- **Action taken:** Parsed each format separately into a new column, `start_time_parsed`, while retaining the original column.
- **Justification:** Although dates are not required in the final analysis, parsing them helps validate the dataset's time coverage.
- **Evidence:** Zero unparseable dates. All dates fall between 1 and 31 March 2025.

#### 4. Missing Trip Durations
- **Problem:** 30 missing values in `duration_min`.
- **Action taken:** Excluded records with missing durations from average-duration calculations.
- **Justification:** Missing values cannot contribute to the average. Imputing them would introduce assumptions about the variable being measured.
- **Evidence:** 30 records, approximately 4.9% of the original dataset.

#### 5. Extreme Trip Durations
- **Problem:** Three extreme durations of 1,440, 1,440 and 2,880 minutes.
- **Action taken:** Excluded trips with durations exceeding 120 minutes.
- **Justification:** These values are substantially above the 99th percentile of 52 minutes. Their exact multiples of 24 hours suggest possible unreturned bikes or logging errors. The 120-minute threshold is an analytical judgement based on the distribution.
- **Evidence:** Three records excluded, approximately 0.5% of the original dataset.

#### 6. Table Joins
- **Problem:** Trip records need to be linked to bike types and station zones.
- **Action taken:** Performed inner joins with `bikes` using `bike_id` and `stations` using `start_station_id` and `station_id`. Added row-count assertions after each join.
- **Justification:** Previous checks confirmed that all foreign keys matched existing records and that lookup-table keys were unique. Therefore, the joins should not remove or multiply trip records.
- **Evidence:** 600 rows expected after each join, before excluding missing durations and extreme durations.

#### 7. Missing `capacity` and Inconsistent `user_type`
- **Problem:** One missing `capacity` value and inconsistent casing or whitespace in `user_type`.
- **Action taken:** Left both columns unchanged.
- **Justification:** Neither variable is required to answer the research question, so cleaning them would not contribute to the current analysis.
- **Evidence:** No changes made.

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

####Expectations

1. Trips_filtered has 567 rows. bikes and stations have unique keys and 0 orphans, so I expect 567 rows after the bikes join and 567 after the stations join (one match per trip, nothing dropped or duplicated).
2. Final table grain: one row = one combination of bike_type and zone of the start station, with the number of trips and the mean duration in minutes.
3. I expect at most 10 rows (2 bike types × 5 zones), and n_trips should sum to 567.
4. Before looking, my hypothesis is that electric trips may be longer or shorter than standard ones, and I will check whether the direction is the same in every zone.


In [18]:
# Overall by bike type (first half of the question)
overall = (
    joined_df.groupby("bike_type")["duration_min"]
    .agg(n_trips="count", avg_duration_min="mean", median_duration_min="median")
    .round(2)
)
display(overall)

# Believability check 1: weighted average of group means should equal the overall mean
overall_mean = joined_df["duration_min"].mean()
weighted_mean = (agg_results["avg_duration_min"] * agg_results["n_trips"]).sum() / agg_results["n_trips"].sum()
print("overall mean:", round(overall_mean, 2), "| weighted mean from table:", round(weighted_mean, 2))
print("agree within rounding:", abs(overall_mean - weighted_mean) < 0.01)

# Believability check 2: do pandas and SQL agree group by group?
cmp = agg_results.merge(sql_results, on=["bike_type", "zone"], suffixes=("_pd", "_sql"))
print("groups matched:", len(cmp), "of", len(agg_results))
print("counts identical:", (cmp["n_trips_pd"] == cmp["n_trips_sql"]).all())
print("averages identical:", ((cmp["avg_duration_min_pd"] - cmp["avg_duration_min_sql"]).abs() < 0.01).all())

# Gap by zone (standard minus electric)
gap = agg_results.pivot(index="zone", columns="bike_type", values="avg_duration_min")
gap["standard_minus_electric"] = (gap["standard"] - gap["electric"]).round(2)
display(gap)

,n_trips,avg_duration_min,median_duration_min
bike_type,,,
electric,161,14.80,13.0
standard,406,19.33,17.0


overall mean: 18.04 | weighted mean from table: 18.05
agree within rounding: True
groups matched: 10 of 10
counts identical: True
averages identical: True


bike_type,electric,standard,standard_minus_electric
zone,,,
Centre,14.51,18.93,4.42
East,15.73,19.50,3.77
North,15.96,20.36,4.40
South,13.42,19.47,6.05
West,15.68,19.82,4.14


# Results and Interpretation

## 1. Overall Findings

Electric bikes are **not associated with longer trips in terms of duration**. Instead, electric bike trips have shorter average durations than standard bike trips.

Across all 567 cleaned trips:

- **Electric bikes:** Average duration of 14.80 minutes across 161 trips, with a median of 13 minutes.
- **Standard bikes:** Average duration of 19.33 minutes across 406 trips, with a median of 17 minutes.
- **Difference:** Standard bike trips average approximately 4.5 minutes longer than electric bike trips.

Overall, the results suggest that electric bike trips tend to have shorter durations than standard bike trips.

## 2. Does the Pattern Differ by Zone?

The overall pattern remains consistent across all five zones: **standard bike trips have longer average durations than electric bike trips in every zone.**

However, the size of the difference varies between zones.

### Centre
- Electric bike average: 14.51 minutes
- Standard bike average: 18.93 minutes
- Difference: 4.42 minutes

### East
- Electric bike average: 15.73 minutes
- Standard bike average: 19.50 minutes
- Difference: 3.77 minutes

### North
- Electric bike average: 15.96 minutes
- Standard bike average: 20.36 minutes
- Difference: 4.40 minutes

### South
- Electric bike average: 13.42 minutes
- Standard bike average: 19.47 minutes
- Difference: 6.05 minutes

### West
- Electric bike average: 15.68 minutes
- Standard bike average: 19.82 minutes
- Difference: 4.14 minutes

### Key Observation

- **Largest difference:** South, where standard bike trips average approximately 6.05 minutes longer.
- **Smallest difference:** East, where standard bike trips average approximately 3.77 minutes longer.
- **Other zones:** Centre, North and West have relatively similar differences, ranging from approximately 4.14 to 4.42 minutes.

Therefore, the direction of the relationship does not change across zones, although the magnitude of the difference varies.

## 3. Data Validation and Reliability Checks

All validation checks passed.

- **Join integrity:** Both joins retained 567 rows, indicating that no trips were dropped or duplicated during the joins.
- **Group completeness:** The final summary contains 10 groups, representing two bike types across five zones.
- **Trip counts:** The `n_trips` values sum to 567, matching the total number of cleaned trips.
- **Missing averages:** No group has a null average duration.
- **Independent verification:** An independent DuckDB SQL query produced the same trip counts and average durations as the pandas analysis across all 10 groups.
- **Weighted average check:** The weighted average of the group means is approximately 18.05 minutes, consistent with the overall mean of 18.04 minutes after rounding.

These checks support the internal consistency of the analysis.

## 4. Caveats and Limitations

### 4.1 Unequal Group Sizes

Some groups contain fewer observations than others.

- East electric: 11 trips.
- East standard: 20 trips.
- Centre: 296 trips, representing approximately 52% of all cleaned trips.

The East averages are therefore less reliable than those from larger groups. Additionally, the Centre zone has a substantial influence on the overall results because it accounts for more than half of the trips.

### 4.2 Differences Between Means and Medians

In the East zone, the electric bike median is 18 minutes, compared with 16.5 minutes for standard bikes. This is the opposite of the pattern observed in the average durations.

With only 11 electric bike trips and 20 standard bike trips, this difference should be interpreted cautiously. The small sample sizes may contribute to the discrepancy, although further analysis would be needed to determine the reason.

In the other four zones, the medians follow the same direction as the means, with standard bike trips having longer durations.

### 4.3 Duration Does Not Measure Distance

Trip duration should not be interpreted as trip distance.

Electric bikes may allow riders to travel faster, meaning that shorter durations do not necessarily indicate shorter journeys in terms of distance.

Since the dataset does not provide trip distances, the analysis can only conclude that electric bike trips have shorter recorded durations on average. It cannot establish whether the journeys themselves are shorter.

### 4.4 Data Cleaning Assumptions

The analysis involved several cleaning decisions:

- Excluded 30 trips with missing `duration_min` values, representing approximately 5% of the original dataset.
- Excluded three trips with durations exceeding 120 minutes.
- Treated `E-bike` as equivalent to `electric`.

These decisions were made to improve data consistency and reduce the influence of potentially erroneous observations. However, the 120-minute threshold is a judgement-based decision and may affect the results.

### 4.5 Missing Data Patterns Not Investigated

The analysis did not examine whether missing trip durations were concentrated within particular bike types or station zones.

If missingness is more common in certain groups, excluding those records could introduce bias into the comparisons.

Further analysis should examine the distribution of missing values across bike types and zones.

### 4.6 No Statistical Significance Testing

The analysis compares descriptive statistics, including means and medians, but does not include statistical significance tests.

Therefore, the observed differences should not automatically be interpreted as statistically significant. Further testing would be required to assess whether the differences between bike types, and the variation across zones, are likely to reflect genuine patterns rather than random variation.

## 5. Overall Conclusion

The analysis indicates that electric bike trips have shorter average durations than standard bike trips across all five station zones. The difference ranges from approximately 3.77 minutes in the East zone to 6.05 minutes in the South zone.

Although the pattern is consistent across zones, the findings should be interpreted with consideration of unequal group sizes, differences between means and medians, data-cleaning assumptions, and the absence of statistical significance testing.

Most importantly, **the results demonstrate a difference in trip duration, not necessarily trip distance**, and do not establish that bike type causes the observed differences.

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

####How I’ll check the answer
1. Count check: reconcile every row from 612 raw trips to 567 in the final table, and confirm no trip_id appears twice.
2. Spot check: pick 3 random trips from the final table and look each one up by hand in the raw tables to confirm its bike type and zone.
3. Recalculation: rebuild the electric averages for East and South from the raw data, using explicit lists of labels and IDs instead of my cleaning code. I expect 11 trips and 15.73 min for East, and 24 trips and 13.42 min for South.
4. Sensitivity: re-run the comparison with different duration cutoffs, and check whether the missing durations cluster in one group.


In [21]:
raw_n = len(tables["trips"])
dedup_n = len(tables["trips"].drop_duplicates())
missing_n = tables["trips"].drop_duplicates()["duration_min"].isna().sum()
outlier_n = (tables["trips"].drop_duplicates()["duration_min"] > 120).sum()

print(f"raw {raw_n} - duplicates {raw_n - dedup_n} - missing {missing_n} - outliers {outlier_n} = {raw_n - (raw_n - dedup_n) - missing_n - outlier_n}")
print("final table n_trips sum:", agg_results["n_trips"].sum())
print("duplicate trip_ids in joined_df:", joined_df["trip_id"].duplicated().sum(), "(expected 0)")

raw 612 - duplicates 12 - missing 30 - outliers 3 = 567
final table n_trips sum: 567
duplicate trip_ids in joined_df: 0 (expected 0)


In [22]:
sample = joined_df.sample(3, random_state=1)
for _, r in sample.iterrows():
    print("FINAL:", r["trip_id"], r["bike_id"], r["bike_type"], r["start_station_id"], r["zone"], r["duration_min"])
    display(tables["trips"][tables["trips"]["trip_id"] == r["trip_id"]])      # raw trip row(s)
    display(tables["bikes"][tables["bikes"]["bike_id"] == r["bike_id"]])      # raw bike label
    display(tables["stations"][tables["stations"]["station_id"] == r["start_station_id"]])  # raw zone

FINAL: T0580 B016 standard S04 West 20.0


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
590,T0580,B016,S04,S02,2025-03-30 11:33:00,20.0,Casual


,bike_id,bike_type,purchase_year
15,B016,Standard,2023


,station_id,station_name,zone,capacity
3,S04,University Gate,West,28.0


FINAL: T0099 B028 standard S08 South 11.0


...

,station_id,station_name,zone,capacity
7,S08,Riverside Walk,South,14.0


FINAL: T0144 B012 electric S02 Centre 20.0


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
144,T0144,B012,S02,S12,2025-03-08 15:42:00,20.0,MEMBER


,bike_id,bike_type,purchase_year
11,B012,Electric,2022


,station_id,station_name,zone,capacity
1,S02,Railway Station,Centre,23.0


In [23]:
electric_labels = ["Electric ", "electric", "Electric", "E-bike"]   # typed by hand, not via str.lower()
elec_bikes = set(tables["bikes"].loc[tables["bikes"]["bike_type"].isin(electric_labels), "bike_id"])

for zone_name in ["East", "South"]:
    zone_stations = set(tables["stations"].loc[tables["stations"]["zone"] == zone_name, "station_id"])
    t = tables["trips"].drop_duplicates()
    t = t[t["bike_id"].isin(elec_bikes) & t["start_station_id"].isin(zone_stations)]
    t = t[t["duration_min"].notna() & (t["duration_min"] <= 120)]
    print(zone_name, "electric: n =", len(t), "| mean =", round(t["duration_min"].mean(), 2))

East electric: n = 11 | mean = 15.73
South electric: n = 24 | mean = 13.42


In [24]:
base = (trips_cleaned
        .merge(bikes_cleaned[["bike_id", "bike_type"]], on="bike_id")
        .merge(tables["stations"][["station_id", "zone"]], left_on="start_station_id", right_on="station_id"))

# Does the conclusion depend on my 120-minute cutoff?
for cutoff in [60, 90, 120, 180, 3000]:
    d = base[base["duration_min"].notna() & (base["duration_min"] <= cutoff)]
    m = d.groupby("bike_type")["duration_min"].mean().round(2)
    print(f"cutoff {cutoff:>4}: electric {m['electric']} | standard {m['standard']} | n = {len(d)}")

# Do missing durations cluster?
base["missing"] = base["duration_min"].isna()
display(base.groupby("bike_type")["missing"].agg(["sum", "mean"]).round(3))
display(base.groupby("zone")["missing"].agg(["sum", "mean"]).round(3))

cutoff   60: electric 14.8 | standard 19.04 | n = 565
cutoff   90: electric 14.8 | standard 19.33 | n = 567
cutoff  120: electric 14.8 | standard 19.33 | n = 567
cutoff  180: electric 14.8 | standard 19.33 | n = 567
cutoff 3000: electric 14.8 | standard 33.27 | n = 570


,sum,mean
bike_type,,
electric,13,0.075
standard,17,0.040


,sum,mean
zone,,
Centre,16,0.051
East,5,0.139
North,1,0.016
South,2,0.023
West,6,0.059


# Validation Checks and Robustness Analysis

## Check 1: Count Reconciliation

**Purpose:** Verify that every trip is accounted for during data cleaning.

- Initial dataset: 612 trips.
- Exact duplicates removed: 12 trips.
- Missing durations excluded: 30 trips.
- Extreme durations excluded (over 120 minutes): 3 trips.
- Final dataset: 567 trips.

**Calculation:**

612 − 12 − 30 − 3 = 567 trips.

**Result:** The calculated total matches the sum of `n_trips` in the final summary table. No `trip_id` values are duplicated in the final dataset.

**Conclusion:** All records are accounted for, and the final trip count is consistent with the cleaning decisions.

---

## Check 2: Manual Spot Check

**Purpose:** Verify that the final dataset correctly represents the original records.

I selected three trips at random and traced them back to the raw tables:

- `T0580`
- `T0099`
- `T0144`

For all three trips, the bike type, station zone and duration in the final dataset matched the corresponding records in the raw tables.

**Result:** All three spot checks passed.

**Limitation:** All three selected trips had clean labels in the raw data. Therefore, this check did not directly test inconsistent labels such as `Electric ` and `E-bike`. These cases were examined separately in Check 3.

---

## Check 3: Independent Recalculation from Raw Data

**Purpose:** Verify the label standardisation and table joins independently of the main cleaning code.

I independently rebuilt two groups, East electric and South electric, directly from the raw tables. I used manually specified lists of electric-bike labels and station IDs rather than relying on the original label-cleaning code.

**Results:**

- **East electric:** 11 trips, with an average duration of 15.73 minutes.
- **South electric:** 24 trips, with an average duration of 13.42 minutes.

Both results matched the main summary table.

**Conclusion:** This provides additional evidence that the label standardisation and joins are correct.

**Limitation:** This check does not independently validate the duplicate-removal decision or the 120-minute cutoff, because the same cleaning rules were applied.

---

## Check 4: Sensitivity to the Duration Cutoff

**Purpose:** Determine whether the main finding changes when different duration thresholds are applied.

The comparison below shows the average trip duration by bike type under different cutoffs.

### Results by Cutoff

**Cutoff: 60 minutes**
- Electric average: 14.80 minutes.
- Standard average: 19.04 minutes.
- Total trips: 565.

**Cutoff: 90 minutes**
- Electric average: 14.80 minutes.
- Standard average: 19.33 minutes.
- Total trips: 567.

**Cutoff: 120 minutes (main analysis)**
- Electric average: 14.80 minutes.
- Standard average: 19.33 minutes.
- Total trips: 567.

**Cutoff: 180 minutes**
- Electric average: 14.80 minutes.
- Standard average: 19.33 minutes.
- Total trips: 567.

**No cutoff (up to 3,000 minutes)**
- Electric average: 14.80 minutes.
- Standard average: 33.27 minutes.
- Total trips: 570.

### Interpretation

- The results are stable between 90 and 180 minutes because no trips fall within the relevant duration range.
- The 60-minute cutoff removes only two standard-bike trips and has little effect on the overall comparison.
- The three extreme trips (1,440, 1,440 and 2,880 minutes) are all standard-bike trips.
- Retaining these outliers increases the standard-bike average to 33.27 minutes, substantially exaggerating the difference between bike types.

**Conclusion:** The main finding is robust to reasonable changes in the duration cutoff. Excluding the three extreme trips prevents potentially erroneous durations from disproportionately influencing the standard-bike average.

---

## Check 5: Investigation of Missing Durations

**Purpose:** Determine whether missing `duration_min` values are concentrated in particular bike types or station zones.

### Missing Durations by Bike Type

**Electric bikes**
- Missing values: 13.
- Missing rate: 7.5%.

**Standard bikes**
- Missing values: 17.
- Missing rate: 4.0%.

### Missing Durations by Station Zone

**East**
- Missing values: 5.
- Missing rate: 13.9%.

**Centre**
- Missing values: 16.
- Missing rate: 5.1%.

**West**
- Missing values: 6.
- Missing rate: 5.9%.

**South**
- Missing values: 2.
- Missing rate: 2.3%.

**North**
- Missing values: 1.
- Missing rate: 1.6%.

### Interpretation

The missingness is not evenly distributed across the dataset.

- Electric bikes have a higher missing rate than standard bikes (7.5% compared with 4.0%).
- East has the highest missing rate by zone, with five missing durations out of approximately 36 trips.

For the overall comparison, the missing electric-bike durations would need to average approximately 75 minutes each to eliminate the observed difference between bike types, assuming the observed standard-bike average remains unchanged. This is substantially above the 99th percentile of 52 minutes, making this scenario appear unlikely, although it cannot be ruled out from the available data alone.

However, the missing values remain a more important limitation for the East zone. Excluding five trips from an already relatively small group weakens the reliability of its results.

**Limitation:** Missingness was examined separately by bike type and by zone. Therefore, it is not yet known whether the five missing durations in East are concentrated among electric bikes, standard bikes, or both.

---

## 6. Overall Conclusion

The count reconciliation, manual spot checks, independent recalculation and sensitivity analysis support the consistency of the main results.

The analysis indicates that **electric bike trips have shorter average durations than standard bike trips overall and across all five station zones**. The finding also remains stable under reasonable changes to the duration cutoff.

However, several limitations remain:

- **Small East groups:** East has relatively few observations and the highest missing-data rate, making its results less reliable.
- **Duration versus distance:** Shorter trip durations do not necessarily mean shorter travel distances.
- **Missing data:** Missing durations are more common among electric-bike trips and in the East zone, potentially introducing bias.
- **No significance testing:** The analysis is descriptive and does not establish whether the observed differences are statistically significant.

Overall, the results provide consistent descriptive evidence of differences in trip duration between electric and standard bikes, but further statistical testing and investigation of missing data would strengthen the conclusions.

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

### Prompt 1: Data Cleaning

**Prompt used**

> I have pandas DataFrames `trips`, `bikes`, and `stations`. Findings: 12 exact duplicate rows in `trips`; `bike_type` has variants (`Standard`, `standard`, `STANDARD`, `Electric`, `Electric `, `electric`, `E-bike`); `duration_min` has missing values and three outliers over 120 minutes; `start_time` has two formats (ISO and `dd/mm/yyyy`).
>
> Write short, separate cells that:
> 1. Drop exact duplicates.
> 2. Standardise `bike_type` to `standard` or `electric`.
> 3. Parse both date formats.
> 4. Drop missing durations and durations over 120 minutes.
> 5. Join to `bikes` and `stations`.
>
> Print row counts after each step and assert that the joins do not change the row count.

**Accepted**
- The five-step structure with row counts after each step (612 → 600 → 567).
- Parsing both date formats separately and combining them.
- Preserving the original `start_time` column and creating `start_time_parsed`.
- Using inner joins with row-count assertions to detect unexpected row loss or duplication.

**Changed or added by me**
- I identified the data quality issues through my own inspection rather than asking AI to find them.
- I chose the 120-minute threshold based on the distribution: the 99th percentile was 52 minutes, and the three extreme durations were substantially higher.
- I assumed `E-bike` means electric and documented this assumption.
- I checked the arithmetic myself: 600 − 30 missing durations − 3 outliers = 567, confirming that the exclusions do not overlap.

**Rejected**
- Nothing in this step.
- I did not impute missing durations because doing so would introduce invented values into the variable being measured.

### Prompt 2: Aggregation

**Prompt used**

> I have a cleaned pandas DataFrame `joined_df` containing 567 rows. It was built from `trips_filtered` (duplicates removed, missing durations and durations over 120 minutes excluded), joined to `bikes_cleaned` on `bike_id` and to `stations` on `start_station_id`.
>
> Each row represents one trip. Relevant columns are `trip_id`, `duration_min` (float, minutes), `bike_type` (`standard` or `electric`, already standardised), and `zone` (the starting station zone: Centre, North, South, East or West).
>
> Write Python code that produces one row per combination of `bike_type` and `zone`, with the columns `n_trips`, `avg_duration_min` (mean, rounded to two decimal places), and `median_duration_min`.
>
> Sort by `zone` and then `bike_type`. Print checks to confirm that:
> 1. The number of rows is at most 10.
> 2. `n_trips` sums to `len(joined_df)`.
> 3. No group has a null average.
>
> Use pandas `groupby` and `agg`. Keep it to one short cell with comments.

**Accepted**
- The `groupby` and aggregation code.
- The three validation checks.
- The output matched my expectations: at most 10 groups and a total trip count of 567.

**Added by me**
- I included the median because the mean can be affected by skewed distributions.
- This proved useful because the East zone's median results point in the opposite direction to its mean results.

### Prompt 3: SQL Cross-Check

**Prompt used**

> Using DuckDB, I have registered `trips_filtered` as `trips_clean`, `bikes_cleaned` as `bikes_clean`, and the original stations table as `stations`.
>
> Write a SQL query using my helper function `q()` that joins `trips_clean` to `bikes_clean` on `bike_id`, and to `stations` using `start_station_id = station_id`.
>
> Return `bike_type`, `zone`, `COUNT(*) AS n_trips`, and `ROUND(AVG(duration_min), 2) AS avg_duration_min`. Group by `bike_type` and `zone`, and order by `zone` followed by `bike_type`.

**Accepted**
- The SQL query generated by AI.
- The results matched the pandas summary across all 10 groups, for both trip counts and average durations.

**Limitation identified by me**

This check uses the same cleaned DataFrames as the pandas analysis. Therefore, it verifies the aggregation independently but cannot identify errors introduced during the original cleaning process.

To address this limitation, I added further checks using the raw data in Section 4.

### Prompt 4: Verification Code

**Prompt used**

Insert the exact request sent to the AI assistant here. I received help drafting the verification cells, so I will identify the AI tool actually used.

**Accepted**
- Count reconciliation to confirm that all records are accounted for.
- Random spot checks of three trips against the raw tables.
- Independent recalculation of East electric and South electric using manually specified label lists.
- Sensitivity analysis using different duration cutoffs.
- An investigation of missing-duration rates by bike type and station zone.

**What I concluded from the checks**
- The spot check covered trips with already-standardised labels, so it did not directly verify messy labels such as `Electric ` and `E-bike`.
- The independent recalculation tested label standardisation and joins using the raw tables.
- The recalculation reused my duplicate-removal and 120-minute cutoff rules, so it did not independently validate those decisions.

These limitations were documented when interpreting the reliability of the results.

### Where the AI Output Needed Correcting

**1. Incorrect assumption about date formats**

An early note assumed that `start_time` was entirely in `dd/mm/yyyy` format.

My own inspection showed that 514 of 612 records did not parse under that format alone, indicating that most records used ISO formatting. I parsed both formats separately and confirmed that there were zero unparseable dates.

**2. Incorrect variable name**

The first overall-comparison cell referred to a variable named `summary`, which did not exist in my notebook.

I corrected the variable name to `agg_results`.

**3. Weighted mean comparison**

An early version compared the weighted mean of the group averages with the overall mean using exact equality, despite the group averages being rounded to two decimal places.

I changed the comparison to use a tolerance of 0.01. The weighted mean was approximately 18.05 minutes and the overall mean was approximately 18.04 minutes, so they agree within rounding.

**4. Inaccurate wording**

An early note stated that the two means were equal.

I changed the wording to **"agree within rounding"** because the values were close but not exactly identical.

---

### What I Did Not Use AI For

- Identifying the data quality issues and deciding how to address them.
- Setting expectations before each analysis step, including expected row counts and the structure of the final summary table.
- Choosing and justifying the 120-minute duration cutoff based on the observed distribution.
- Interpreting the results and identifying limitations.
- Recognising that the East zone has small group sizes and that duration does not necessarily represent distance.
- Identifying that missing durations are more common among electric bike trips (7.5%) than standard bike trips (4.0%).

**Overall reflection**

I used AI to assist with writing and structuring code, but I checked the outputs, corrected issues, conducted additional validation, and made the analytical decisions myself. I also documented the limitations of each verification method rather than treating agreement between outputs as proof that every decision was correct.

## 6. What I could not verify

*What remains unchecked? What would you do next?*

## 1. Compare Trips on the Same Routes

Restrict the comparison to trips sharing the same starting and ending station pair, then compare electric and standard bike durations on those routes.

**Purpose:** This would help investigate whether the observed difference persists when the routes are comparable, reducing the influence of differences in journey distance.

## 2. Conduct Statistical Significance Testing

Use an appropriate statistical method, such as Welch's t-test, or bootstrap confidence intervals, to assess differences in average durations overall and within each zone.

**Purpose:** This would help quantify uncertainty and assess whether the observed differences are distinguishable from random variation. Small group sizes should be considered when interpreting the results.

## 3. Investigate Missingness Within Each Group

Examine missing-duration rates by bike type within each station zone. Also investigate whether missing trips share other characteristics.

**Purpose:** This would help determine whether missingness is concentrated in particular combinations of bike type and zone, potentially affecting the comparisons.

## 4. Investigate Potential Confounding Factors

Repeat the comparisons across relevant user groups, times of day and bike ages, where reliable data is available.

Potential variables include:
- Cleaned `user_type`.
- Time of day.
- Bike age, if a reliable measure can be obtained.

**Purpose:** This would help determine whether the observed association remains when other relevant factors are considered.

## 5. Confirm Data-Cleaning Assumptions

Consult the data owner to verify:
- Whether `E-bike` is intended to mean electric.
- Whether the 24-hour and 48-hour trips represent logging errors or genuine journeys.
- Whether the 12 exact duplicate records are accidental duplicates.

**Purpose:** This would provide stronger evidence for the cleaning decisions rather than relying entirely on assumptions.

## 6. Extend the Validation Checks

Expand the manual spot checks to include trips with inconsistent bike-type labels, particularly `Electric ` and `E-bike`. Also investigate very short trips and trips where the starting and ending stations are identical.

**Purpose:** This would strengthen the validation of label standardisation and help identify additional potentially implausible records.

## Overall Reflection

The analysis provides consistent descriptive evidence that electric bike trips have shorter average durations than standard bike trips in the available dataset. However, missing data, small groups, potential confounding factors and unverified cleaning assumptions limit the strength of the conclusions.

The next steps would help establish how robust the observed differences are, whether they remain after accounting for other factors, and how confidently the findings can be generalised.